# 06 - Neural network

**Task:** predict how many packets will be waiting in a switch's queue 20 ms from now, using its recent history.

This notebook is a copy of `01_quickstart.ipynb`. Only the model in step 6 is different.

## 1. Setup

Find the project folder and import what we need. Run the cells from top to bottom with Shift+Enter.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))   # so that "from src..." works

import matplotlib.pyplot as plt
import pandas as pd

from src.evaluate import add_result, score

## 2. Load the tables

`train` is for teaching the model and `val` is for checking it. There is also a `test` table, but it stays closed until the very end (see `TASKS.md`).

In [ ]:
train = pd.read_parquet(ROOT / "data/tables/train.parquet")
val = pd.read_parquet(ROOT / "data/tables/val.parquet")
print(len(train), "training rows,", len(val), "validation rows")

## 3. Look at 5 rows

Each row is one moment of one run. `y` is what we want to predict. `lag_1` is the queue now, `lag_2` is the queue one sample (2 ms) ago, and so on. `README.md` explains all columns.

In [ ]:
train.head()

## 4. Plot one run's queue

Each dot is one row of `run_01`. The queue is often empty and sometimes fills up quickly. Small holes are moments next to lost samples, which are not in the table.

In [ ]:
run = train[train["run_id"] == "run_01"]

plt.figure(figsize=(12, 3))
plt.plot(run["t_ms"] / 1000, run["lag_1"], ".", markersize=2, color="#2a78d6")
plt.axhline(64, color="grey", linestyle="--", linewidth=1)
plt.text(90, 65, "queue full (64)", color="grey", ha="right")
plt.ylim(-2, 72)
plt.xlabel("time (seconds)")
plt.ylabel("queue length (packets)")
plt.title("run_01")
plt.show()

## 5. Choose the features

The features are all columns except `run_id` and `t_ms` (they only say where the row comes from) and `y` (the answer).

In [ ]:
features = [c for c in train.columns if c not in ["run_id", "t_ms", "y"]]
X_train, y_train = train[features], train["y"]
X_val, y_val = val[features], val["y"]
features

## 6. Train a neural network

A neural network passes the features through layers of small units that each add up their inputs. It learns better when all features have a similar size, so we scale them first. This version uses scikit-learn's `MLPRegressor` with default settings. In a test it took about 4 minutes.

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model = make_pipeline(StandardScaler(), MLPRegressor(random_state=0))
model.fit(X_train, y_train)

predictions = {"neural_net": model.predict(X_val)}   # name -> predictions for val

## 7. Score it on val

MAE and RMSE are in packets, and lower is better. `add_result` adds one row to `results/results.csv`, the table that compares all models. Every time you run this cell it adds a row again.

In [ ]:
for name, y_pred in predictions.items():
    scores = score(y_val, y_pred)
    add_result(name, "val", scores)
    print(f"{name}: MAE {scores['MAE']:.2f} packets, RMSE {scores['RMSE']:.2f} packets")

pd.read_csv(ROOT / "results/results.csv")

## 8. Plot predicted vs actual

Left: every validation row. Dots on the grey line are perfect predictions. Right: 5 seconds of one validation run.

In [ ]:
for name, y_pred in predictions.items():
    fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4), width_ratios=[1, 2])

    left.scatter(y_val, y_pred, s=4, alpha=0.1, color="#2a78d6")
    left.plot([0, 64], [0, 64], color="grey", linewidth=1)   # a perfect prediction
    left.set_xlabel("actual (packets)")
    left.set_ylabel("predicted (packets)")
    left.set_title(f"{name}: all validation rows")

    part = val.assign(y_pred=y_pred)
    part = part[(part["run_id"] == "run_08") & part["t_ms"].between(15_000, 20_000)]
    right.plot(part["t_ms"] / 1000, part["y"], ".", markersize=4, color="#2a78d6", label="actual")
    right.plot(part["t_ms"] / 1000, part["y_pred"], ".", markersize=4, color="#eb6834", label="predicted")
    right.set_xlabel("time (seconds)")
    right.set_ylabel("queue length 20 ms later (packets)")
    right.set_title(f"{name}: 5 seconds of run_08")
    right.legend()

    plt.tight_layout()
    plt.show()

## TODO: tuning

- [ ] `early_stopping=True` (stops when it no longer improves, usually much faster)
- [ ] `hidden_layer_sizes`: try (32,), (100,), (64, 64)
- [ ] `alpha`: try 0.0001, 0.001, 0.01
- [ ] `learning_rate_init`: try 0.0005, 0.001, 0.005
- [ ] optional: an LSTM on `lag_10` ... `lag_1` (needs PyTorch or TensorFlow, so agree with the group first)

Teach every version on `train` and compare on `val`. Do not open the test table.

In [ ]:
# TODO: try other settings here.
# Fit on train, score on val, and add_result(...) for the versions you want to keep.
# Give each version its own name, for example "neural_net 64-64".

## Your notes

*(Write here, in plain words.)*

- What I tried:
- What worked best (settings and val scores):
- What did not help:
- Moments where the model is clearly wrong (look at the plots):
